# LongFlow — Gate Night 1 (pre-75K blocking checks)

Runtime: **L4 GPU**. Plan: `review-pre75k-2.md`. Cheapest/most decisive first;
every cell independent after cold start. What each answers:

| cell | check | decides |
|---|---|---|
| 2 | 20K/80K standardization stats equality (2 min) | retro-validates the dispersion table |
| 3 | teacher determinism given RNG state | whether the paired-map schema is even possible |
| 4 | teacher-vs-teacher reseed floor (audio out) | the metric ceiling any student can reach |
| 5 | sampler A/B renders, 20K head (audio out) | how much of the 0.088 WER is sampler |
| 6 | batched vs unbatched condition parity | whether left-padded capture pollutes the cache |
| 7 | 20-min TEACHER endurance (audio out, LISTEN) | whether long captures would distill teacher degradation |

Colab generates; Mac analyzes. Download `gate_night1_bundle.zip` at the end.


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, sys, time
import soundfile as sf

if not os.path.exists("/content/LongFlow/src"):
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone LongFlow (private: use a token) or drag longflow_bundle.zip + unzip"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.cache.capture import SampleCapture, BatchedSampleCapture, save_utterance
from src.flow_head.cfm import euler_sample, heun_sample
from src.flow_head.trainer import load_checkpoint

CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
TRAIN_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_cache"
OUT = "/content/gate_night1"
os.makedirs(OUT, exist_ok=True)

head20, mean20, std20 = load_checkpoint(f"{CKPT_DIR}/full10k_20k.pt")  # <- fix name if needed
head20 = head20.to("cuda")
FRAME_ID = processor.tokenizer.convert_tokens_to_ids("<|vision_pad|>")

def decode_latents(z):  # [T, d_latent] head-space -> waveform (verified path)
    sc, bi = model.model.speech_scaling_factor, model.model.speech_bias_factor
    z = z.to("cuda", torch.bfloat16) / sc - bi
    out = model.model.acoustic_tokenizer.decode(z.unsqueeze(0))
    return (out[0] if isinstance(out, tuple) else out).detach().float().cpu().numpy().squeeze()

def gen_inputs(text, prompt_wav):
    inputs = processor(text=[f"Speaker 1: {text}\n"], voice_samples=[[prompt_wav]],
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

report = {}
print("READY")


## 2. Stats equality — retro-validates the 20K-vs-80K dispersion table (2 min)

In [ ]:
c20 = torch.load(f"{CKPT_DIR}/full10k_20k.pt", weights_only=True)
c80 = torch.load(f"{CKPT_DIR}/full10k_80k.pt", weights_only=True)
dm = float((c20["latent_mean"] - c80["latent_mean"]).abs().max())
ds = float((c20["latent_std"] - c80["latent_std"]).abs().max())
report["stats_equality"] = {"mean_maxdiff": dm, "std_maxdiff": ds}
print(f"latent_mean max|diff| = {dm:.3e}   latent_std max|diff| = {ds:.3e}")
print("EQUAL (dispersion table valid)" if dm == 0 and ds == 0 else
      "NOT IDENTICAL — dispersion table units differ; paste to Claude")


## 3. Teacher determinism audit — the paired-map schema's core assumption

The upgraded 75K schema captures the teacher's initial noise so training becomes
a paired noise→latent map. That only works if `sample_speech_tokens` is a
DETERMINISTIC function of (RNG state, condition). First print where noise enters
the teacher's sampler, then replay one real (condition, neg_condition) with a
fixed seed twice.


In [ ]:
# where does randomness enter the teacher head?
!grep -n "randn\|Generator\|manual_seed\|noise" /content/VibeVoice/vibevoice/modular/modeling_vibevoice_inference.py | head -30

# grab one real (condition, neg_condition, cfg_scale) from a short generation
grabbed = {}
orig_fn = model.sample_speech_tokens
def grabber(condition, neg_condition=None, cfg_scale=None, **kw):
    if not grabbed:
        grabbed.update(cond=condition.detach().clone(),
                       neg=None if neg_condition is None else neg_condition.detach().clone(),
                       cfg=cfg_scale)
    return orig_fn(condition, neg_condition, cfg_scale, **kw)
model.sample_speech_tokens = grabber
prompt = sorted(glob.glob(f"{EVAL_CACHE_DIR}/*_prompt.wav"))[0]
with torch.inference_mode():
    model.generate(**gen_inputs("The quick brown fox jumps over the lazy dog.", prompt),
                   tokenizer=processor.tokenizer, cfg_scale=1.3)
del model.sample_speech_tokens
print("grabbed:", {k: (tuple(v.shape) if torch.is_tensor(v) else v) for k, v in grabbed.items()})

def call_seeded(seed):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    with torch.inference_mode():
        return orig_fn(grabbed["cond"], grabbed["neg"], grabbed["cfg"]).float()

a, b, c = call_seeded(0), call_seeded(0), call_seeded(1)
same = float((a - b).abs().max()); diff = float((a - c).abs().max())
report["teacher_determinism"] = {"same_seed_maxdiff": same, "diff_seed_maxdiff": diff}
print(f"same seed max|diff| = {same:.3e}   different seed max|diff| = {diff:.3e}")
print("DETERMINISTIC given RNG -> paired-map schema viable" if same < 1e-5 < diff else
      "NOT reproducible under a fixed seed -> schema needs redesign; paste grep output to Claude")


## 4. Teacher-vs-teacher reseed floor — the ceiling any student can reach

Each held-out utterance generated TWICE by the unmodified teacher (different
seeds). Mac computes WER/ECAPA between draws: that number is the metric floor
that legitimate sampling variance imposes. If it's ~0.07 WER, the 20K head's
0.088 is already near the independent-coupling ceiling.


In [ ]:
eval_files = sorted(glob.glob(f"{EVAL_CACHE_DIR}/*.pt"))[:6]
manifest = {}
for f in eval_files:
    d = torch.load(f, weights_only=True)
    tag = d["utt_id"]; manifest[tag] = d["text"]
    prompt = f.replace(".pt", "_prompt.wav")
    for seed, suffix in ((0, "A"), (1, "B")):
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        with torch.inference_mode():
            out = model.generate(**gen_inputs(d["text"], prompt),
                                 tokenizer=processor.tokenizer, cfg_scale=1.3)
        wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
        sf.write(f"{OUT}/{tag}_reseed{suffix}.wav", wav, 24000)
    print(tag, "A/B done")
json.dump(manifest, open(f"{OUT}/reseed_manifest.json", "w"), indent=2)


## 5. Sampler A/B on the 20K head — how much WER is the sampler's fault

heun8 vs euler4/sway0 vs euler16/sway0 renders on the same held-out hiddens.
(eval80k established sim ordering; this pins WER. NOTES never recorded which
sampler produced 0.088.) Mac scores.


In [ ]:
for f in eval_files:
    d = torch.load(f, weights_only=True)
    tag = d["utt_id"]; h = d["hidden"].float().to("cuda")
    for name, fn, nfe in (("heun8", heun_sample, 8), ("euler4", euler_sample, 4),
                          ("euler16", euler_sample, 16)):
        with torch.inference_mode():
            z = fn(head20, h, head20.cfg.d_latent, nfe=nfe, sway=0.0)
        sf.write(f"{OUT}/{tag}_20k_{name}.wav",
                 decode_latents(z * std20.to("cuda") + mean20.to("cuda")), 24000)
    print(tag)


## 6. Batched vs unbatched condition parity — is left-padded capture polluting?

The 75K cache would be captured batch-8 left-padded; inference is unbatched.
Generation is stochastic so token streams may diverge — we compare condition
DISTRIBUTIONS (per-dim mean/std over frames) and report token-stream equality
when it holds. A material distribution shift = pollute-everything bug caught
for $0.30.


In [ ]:
texts = [(torch.load(f, weights_only=True)["utt_id"],
          torch.load(f, weights_only=True)["text"],
          f.replace(".pt", "_prompt.wav")) for f in sorted(glob.glob(f"{EVAL_CACHE_DIR}/*.pt"))[:8]]

# unbatched captures, fixed seed
un = {}
for tag, text, prompt in texts:
    torch.manual_seed(42); torch.cuda.manual_seed_all(42)
    with SampleCapture(model) as cap, torch.inference_mode():
        model.generate(**gen_inputs(text, prompt), tokenizer=processor.tokenizer, cfg_scale=1.3)
    un[tag] = torch.cat([c.reshape(1, -1) for c in cap.conditions]).float()

# batched capture of the same 8, same seed
binputs = processor(text=[f"Speaker 1: {t}\n" for _, t, _ in texts],
                    voice_samples=[[p] for _, _, p in texts], return_tensors="pt", padding=True)
binputs = {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in binputs.items()}
torch.manual_seed(42); torch.cuda.manual_seed_all(42)
with BatchedSampleCapture(model) as bcap, torch.inference_mode():
    bout = model.generate(**binputs, tokenizer=processor.tokenizer, cfg_scale=1.3)
streams = [seq.tolist() for seq in bout.sequences[:, binputs["input_ids"].shape[1]:]]
split = bcap.split_utterances(streams, FRAME_ID)

rows = []
for (tag, _, _), (bh, _) in zip(texts, split):
    u, b = un[tag], bh.float()
    row = {"tag": tag, "frames_unbatched": len(u), "frames_batched": len(b),
           "mean_shift": float((u.mean(0) - b.mean(0)).abs().mean()),
           "std_ratio": float((b.std(0) / u.std(0).clamp_min(1e-6)).mean())}
    if len(u) == len(b):
        row["per_frame_cos"] = float(torch.nn.functional.cosine_similarity(u, b, dim=-1).mean())
    rows.append(row); print(row)
report["condition_parity"] = rows
print("healthy: mean_shift ~0, std_ratio ~1.0, per_frame_cos ~1.0 when frame counts match")


## 7. 20-minute TEACHER endurance — would long captures distill degradation?

Unmodified teacher (DDPM head), ~200 cache sentences (~20 min of audio), with
SampleCapture running — the captured pairs are REUSABLE as the first
long-context training data if the audio is clean. **JOSH LISTENS**: is the
teacher itself still clean at minute 5 / 10 / 15 / 20? If it degrades, note
WHERE — that timestamp caps the harvestable capture length for the 75K mix.
Also the captured conditions feed the E4 context-length OOD probe for free.


In [ ]:
sents = []
for f in sorted(glob.glob(f"{TRAIN_CACHE_DIR}/*.pt"))[-300:]:
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
script = " ".join(sents[:200])
print(f"script: {len(script.split())} words (~20 min)")

prompt = sorted(glob.glob(f"{EVAL_CACHE_DIR}/*_prompt.wav"))[0]
t0 = time.time()
with SampleCapture(model) as cap, torch.inference_mode():
    out = model.generate(**gen_inputs(script, prompt), tokenizer=processor.tokenizer,
                         cfg_scale=1.3, max_new_tokens=12000)
wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
sf.write(f"{OUT}/teacher_endurance20.wav", wav, 24000)
print(f"{time.time()-t0:.0f}s wall, {len(wav)/24000/60:.1f} min audio, {cap.num_frames} frames")
utt = cap.to_utterance("teacher_endurance20", script, meta={"kind": "long_context_probe"})
save_utterance(utt, "/content/drive/MyDrive/longflow_longctx_probe/teacher_endurance20.pt"
               if os.path.isdir("/content/drive/MyDrive/longflow_longctx_probe")
               else (os.makedirs("/content/drive/MyDrive/longflow_longctx_probe", exist_ok=True)
                     or "/content/drive/MyDrive/longflow_longctx_probe/teacher_endurance20.pt"))
zs = utt.latent.float()
n = max(len(zs) // 8, 1)
report["teacher_endurance_latent_std_8seg"] = [round(float(zs[i*n:(i+1)*n].std()), 3) for i in range(8)]
print("teacher latent std, 8 segments:", report["teacher_endurance_latent_std_8seg"])
from IPython.display import Audio, display
display(Audio(f"{OUT}/teacher_endurance20.wav"))


## 8. Bundle

In [ ]:
import zipfile
json.dump(report, open(f"{OUT}/gate_night1_report.json", "w"), indent=2)
with zipfile.ZipFile("/content/gate_night1_bundle.zip", "w") as z:
    for f in glob.glob(f"{OUT}/*"):
        z.write(f, os.path.basename(f))
print("download /content/gate_night1_bundle.zip")
